
# Polars 


Polars es una biblioteca de análisis de datos e implementación de DataFrames de rendimiento ultra-alto escrita en **Rust**. Está orientada al rendimiento, con
ejecución multihilo, expresiones declarativas y un modo de ejecución *lazy*.

* **Paralelismo Nativo**: Mientras que Pandas procesa los DataFrames utilizando un solo hilo de ejecución (single-threaded), Polars divide automáticamente cualquier operación entre todos los núcleos disponibles del procesador.


# Motivación


| Herramienta | Umbral Óptimo de Datos | Modelo de Ejecución | Gestión de Memoria | Caso de Uso Ideal |
| :--- | :--- | :--- | :--- | :--- |
| **Pandas** | < **10 GB** | Monohilo (Eager) | Carga todo en RAM | Análisis exploratorio local en datasets pequeños. |
| **Polars** | <  **100 GB** | Multihilo, Vectorizado en Rust (Lazy/Eager) | Soporta *Streaming* (procesa por lotes sin cargar todo en RAM). | Procesamiento **local** de muy alto rendimiento. |
| **DuckDB** | < **200 GB** | OLAP embebido en C++ (SQL) | *Out-of-core*: Vuelca automáticamente a disco si se agota la RAM. | Consultas SQL analíticas sobre archivos Parquet/CSV locales. |
| **Dask** | **20 - 500 GB** | Distribuido (DataFrames en paralelo) | Extiende Pandas/NumPy fragmentando tareas en grafos (*Lazy*). | Escalar código de Pandas en múltiples núcleos o cluster. |
| **Apache Spark** | **> 100 GB a Petabytes** | Distribuido (*Clusters* multinodo) | Maneja memoria y disco mediante RDDs / DataFrames. | Big Data masivo, ETL empresarial y procesamiento distribuido. |


* **Escala local (< 100 GB)**:
    - Evita **Pandas** si el archivo pesa más del 20-30% de tu RAM (generará fallos Out-Of-Memory).
    - Usa **Polars** o **DuckDB**: aprovechan al máximo todos los núcleos del CPU y procesan conjuntos de datos que superan la memoria RAM mediante evaluación perezosa (Lazy evaluation).
    
* **Escala masiva / Distribuida (> 1 TB)**:
    - **Spark** es el estándar de la industria cuando los datos sobrepasan la capacidad de un solo servidor y deben distribuirse en decenas o cientos de máquinas.
    - **Dask** es una alternativa práctica si tu infraestructura usa Python de forma nativa y necesitas paralelizar tareas que exceden la memoria.   


## Polars

Polars está especialmente orientado a:

- Datasets grandes
- **Procesamiento multihilo** : Mientras que Pandas procesa los DataFrames utilizando un solo hilo de ejecución (single-threaded), Polars puede dividir automáticamente cualquier operación entre todos los núcleos disponibles del procesador (Paralelismo Nativo)
- Flujos de transformación complejos
- Lectura eficiente de archivos de datos distribuidos (Parquet)
- Ejecución *lazy* y optimización automática 
- Procesamiento por lotes o *streaming*.

**Comparación entre Pandas y Polars**

| Pandas | Polars |
|---|---|
| `DataFrame` | `DataFrame` Estructura Bidimensional - **Modo Eager**. Opera en evaluación ansiosa: cada transformación se calcula inmediatamente en la memoria RAM|
| `Series` | `Series` Estructura Unidimensional - **Modo Eager** |
|          | `LazyFrame` no contiene datos cargados en memoria; representa un plan de ejecución. Se genera usando funciones de tipo `scan_*()` o llamando al método `.lazy()` sobre un DataFrame |
| `loc` / `iloc` | Expresiones |
| `groupby` | `group_by` |
| `merge` | `join` |
| `read_csv` | `read_csv` / `scan_csv` |
| Ejecución inmediata | Eager o lazy |


In [ ]:
# Instalación desde un terminal:
# pip install polars pyarrow
# uv add polars pyarrow

import polars as pl
import pandas as pd


print(pl.__version__)


__Primeros ejemplos__

In [ ]:
s = pl.Series("a", [1, 2, 3, 4, 5])
s

In [ ]:
from datetime import datetime

df = pl.DataFrame(
    {
        "integer": [1, 2, 3, 4, 5],
        "date": [
            datetime(2022, 1, 1),
            datetime(2022, 1, 2),
            datetime(2022, 1, 3),
            datetime(2022, 1, 4),
            datetime(2022, 1, 5),
        ],
        "float": [4.0, 5.0, 6.0, 7.0, 8.0],
    }
)

df

__Modos de Ejecución__

* Modo Ansioso (**Eager - DataFrame**): Ejecuta las operaciones en el momento exacto en que se declaran, similar a Pandas.
* Modo Perezoso (**Lazy - LazyFrame**): Construye un **grafo de consulta**. No procesa los datos hasta que se llama a `.collect()`, lo que le permite optimizar el plan de ejecución antes de leer los datos de disco. La operación `collect()` se utiliza para ejecutar una consulta diferida (LazyFrame) y materializar el resultado en memoria como un DataFrame.

In [ ]:
# Creación de un DataFrame desde un diccionario
df = pl.DataFrame(
    {
        "id": [1, 2, 3, 4],
        "usuario": ["Ana", "Carlos", "Elena", "David"],
        "saldo": [1200.50, 850.00, 3100.25, 95.00],
    }
)

# Convertir un DataFrame a LazyFrame
lf = df.lazy()

# Definir transformaciones (no ejecuta nada aún)
consulta = lf.filter(pl.col("saldo") > 500).select(["usuario", "saldo"])

# Ejecutar el plan y obtener el DataFrame final
df_filtrado = consulta.collect()
df_filtrado

In [ ]:
consulta

In [ ]:
type(df_filtrado), type(lf), type(consulta)


# 2. DataFrames, Series y tipos

Polars utiliza un esquema explícito y tipos propios.


In [ ]:
# Creación de un polar Dataframe 
df = pl.DataFrame({
    "nombre": ["Ana", "Luis", "Marta", "Pablo"],
    "edad": [23, 31, 28, 35],
    "ciudad": ["Madrid", "Sevilla", "Madrid", "Valencia"],
    "salario": [2100, 2800, 2500, 3200],
})

df


In [ ]:
print(df.schema)

In [ ]:
print(df.shape)
df.head(2)

Polars realiza una aplicación estricta de esquemas. Cada columna tiene un tipo de datos fijo, y se garantiza que todas las operaciones lo respeten. Esto evita errores que puedan surgir en operaciones de tipado dinámico.

Algunos de tipos de datos que incluye Polars:

| Categoría             | Tipo Polars   | Descripción                                |
| --------------------- | ------------- | ------------------------------------------ |
| Enteros con signo     | `Int8`, `Int16`, `Int32` e `Int64`      |                           |                          |
| Enteros sin signo     | `UInt8` , `UInt16`, `UInt32` y `UInt64`      |               |
| Coma flotante         | `Float32`  ,`Float64`    |         |
| Booleano              | `Boolean`     | Valores `True` o `False`                   |
| Texto                 | `String`      | Cadena de texto UTF-8                      |
|                       | `Utf8`        | Alias histórico de `String`                |
| Temporal         | `Date`   , `Datetime`, `Duration`, `Time`     | |




También dispone de otros tipos:

| Categoría             | Tipo Polars   | Descripción                                |
| --------------------- | ------------- | ------------------------------------------ |
| Binario               | `Binary`      | Datos binarios (`bytes`)                   |
| Categórico            | `Categorical` | Texto codificado por categorías            |
|                       | `Enum`        | Categorías predefinidas                    |
| Estructuras complejas | `List(T)`     | Lista de elementos del tipo `T`            |
|                       | `Array(T, n)` | Array de tamaño fijo                       |
|                       | `Struct`      | Estructura con múltiples campos            |
| Especiales            | `Null`        | Valores nulos                              |
|                       | `Object`      | Objeto Python arbitrario (menos eficiente) |


## Conversión entre Pandas y Polars


In [ ]:
df_pandas = df.to_pandas()
df_desde_pandas = pl.from_pandas(df_pandas)

df_pandas.head()

## Operaciones básicas


| Operación      | Descripción | 
|:----------- |:----------- |
| head(N), tail(N) | Devuelve las  N filas | 
| first()   | Devuelve la primera  fila (LazyFrames)        | 
| describe() | devuelve un dataframe con los primeros estadísticos |Transformación |
| count() |  número de filas |
| collect_schema() |esquema del LazyFrame| 
| schema |esquema del polar dataframe | 




## Collect y show

En Polars,` collect()` es el método que:
* ejecuta el plan de operaciones de un `LazyFrame` 
*  materializa el resultado en memoria
*  devuelve un DataFrame de Polars

La operación `show()` cumple una función similar a `head()` al permitir visualizar las primeras filas de un `LazyFrame`. 

Cuando se trabaja con LazyFrame, es más eficiente aplicar `head()` antes de `collect()` (`lf.head(5).collect()`), ya que el límite forma parte del plan de ejecución y evita procesar más datos de los necesarios.

## Entrada y Salida de Datos (I/O)



### Tipos de archivos

#### Archivos Parquet

`Parquet` es un formato columnar, comprimido y muy eficiente para analítica.

* En vez de guardar fila por fila (como CSV), guarda columna por columna.
* Eso permite leer solo las columnas que necesitas en una consulta.
* Ofrece alta compresión y permite que Polars ejecute `Projection Pushdown` (leer solo las columnas necesarias) y `Predicate Pushdown` (saltarse bloques enteros de datos basándose en metadatos)

#### Archivos Arrow

**Arrow** (IPC / Feather) es un formato columnar en formato binario, optimizado para la máxima velocidad de transferencia e interoperabilidad en memoria.

* En lugar de serializar datos a texto (como CSV) o comprimirlos agresivamente para almacenamiento (como Parquet), organiza los datos en disco con la misma estructura exacta que tienen en la memoria RAM.
* El sistema mapea el archivo directamente a memoria sin consumir ciclos de CPU para deserializar o parsear tipos de datos.
* Ofrece acceso vectorial ultra rápido y permite que Polars ejecute `Projection Pushdown` y comparta datos de forma instantánea entre distintos procesos o lenguajes (como Python, Rust o C++) sin ningún costo de conversión.

**Tabla comparativa** 

| | CSV | Parquet | Arrow (IPC / Feather) |
| :--- | :--- | :--- | :--- |
| **Tipo** | Basado en filas (Texto plano) | Basado en columnas (Binario) | Basado en columnas (Memoria/Binario) |
| **Velocidad Lectura/Escritura** | Lenta (Requiere parsear texto) | Alta (Muy rápida en lectura) | **Ultra rápida** (Zero-copy / Mapeo directo a RAM) |
| **Tamaño en Disco** | Grande (Sin compresión por defecto) | **Muy pequeño** (Compresión Snappy/Zstd eficiente) | Mediano (Optimizado para RAM, no para compresión) |
| **Tipos de Datos** | Todo es texto, requiere inferencia | **Fuerte** (Esquema integrado: guarda tipos de datos, evitando inferencias) | **Fuerte** (Estructura nativa de memoria) |
| **Obliga a leer todo el archivo** | Sí | **No** (Lee solo columnas/filas requeridas) | **No** (Acceso directo por columnas) |
| **Compatibilidad** | Universal (Excel, Scripts, DBs antiguas) | Alta (Ecosistema Big Data: Spark, DuckDB, Polars) | Alta (Interoperabilidad en RAM entre Python, R, C++) |
| **Caso de Uso Ideal** | Intercambio simple de datos pequeños o exportación legible por humanos. | **Almacenamiento permanente / Data Lakes**, análisis analítico (OLAP) masivo. | **Pipelines entre procesos**, datos temporales/caché de alto rendimiento. |

### Lectura y escritura 


**Lectura**

Disponemos de dos modos de lectura donde la  diferencia fundamental  radica en la estrategia de evaluación: Eager (Inmediata) frente a Lazy (perezosa).

* `read_*()` ejecuta una `lectura inmediata` y directa (`Eager`), cargando todo el conjunto de datos desde el disco a la memoria RAM, devolviendo un `DataFrame`. No aprovecha optimizaciones globales y puede agotar la memoria si el archivo es demasiado grande.

* `scan_()*` genera un `plan de ejecución diferido` (`Lazy`) sin cargar datos en memoria al instanciarse, devolviendo en su lugar un `LazyFrame`. Esta estrategia permite que Polars analice toda la consulta antes de ejecutarse para aplicar optimizaciones automáticas —como leer únicamente las columnas necesarias o saltarse filas filtradas— permitiendo procesar datasets que superan la capacidad de la memoria RAM.

**Escritura**

Para guardar o escribir datos a disco en Polars, distinguimos entre si estamos trabajando con un DataFrame (Eager) o con un LazyFrame (Lazy):

* `write_*()` (Eager) : (`write_parquet()`, `write_csv()` o `write_ipc()`) se aplican directamente sobre un `DataFrame` que ya se encuentra totalmente cargado en la memoria RAM. La escritura se realiza de forma inmediata y directa en una sola operación.

* `sink_*()` (Lazy): (`sink_parquet()`, `sink_csv()` o `sink_ipc()`) se aplican sobre un `LazyFrame` para ejecutar la consulta y escribir el resultado en disco mediante `transmisión por lotes` (Streaming). Esto permite procesar y guardar conjuntos de datos masivos que superan la capacidad de la memoria RAM sin necesidad de cargar todo el dataset al mismo tiempo.


#### Lectura

__Ejemplo 1:__

In [ ]:
# Lectura Eager (Carga todo el archivo inmediatamente en RAM)
path = "./datos/polizas.csv"
df_polizas = pl.read_csv(
    path,
    separator=",",
    has_header=True
)
df_polizas.head()


In [ ]:
# lectura super rápida
df_goles = pl.read_parquet("datos/DataSetPartidos.parquet")
df_goles.tail(3)

In [ ]:
# lectura super rápida
df_goles = pl.read_ipc("datos/partidos.arrow")
df_goles.tail(3)

__Atributos de polars dataframes__

* `dtypes` para conocer el tipo de datos de los campos del dataframe.
* `columns`: lista de columnas del dataframe

In [ ]:
# tipos de datos
df_goles.dtypes

In [ ]:
df_goles.columns

__Ejemplo 2:__

In [ ]:
# 2. Lectura Lazy (Recomendado para archivos grandes)
path = './datos/polizas.csv'
lf_polizas = pl.scan_csv(
    path,
    separator=",",
    schema_overrides={"edad": pl.UInt32},  # Forzar tipo de dato explícito
)
lf_polizas

In [ ]:
# Escanear Parquet en modo Lazy
path_parquet = "./datos/DatasetPartidos.parquet"
lf_parquet = pl.scan_parquet(path_parquet)
lf_parquet.collect().head()

In [ ]:
# Escanear arrow en modo Lazy
path_arrow = "./datos/partidos.arrow"
lf_arrow = pl.scan_ipc(path_arrow)
lf_arrow.collect().head(3)

In [ ]:
lf_arrow.explain(optimized=True)

In [ ]:
# mostramos unas pocas filas
lf_arrow.show(5)

Explorar los tipos de datos y columnas con funciones:

* `collect_schema()`
* `collect_schema().dtypes()`
* `collect_schema().names()`

In [ ]:
lf_arrow.collect_schema()

In [ ]:
# tipos de datos
lf_arrow.collect_schema().dtypes()

In [ ]:
lf_arrow.collect_schema().names()

#### Escritura 

Para guardar los datos a disco en Polars, tenemos dos opciones:

**El LazyFrame cabe en memoria RAM**: 
Primero ejecutamos el plan con `.collect()` para obtener un DataFrame y luego usamos los métodos estándar de escritura:

In [ ]:
# Convertir de LazyFrame a DataFrame en RAM
df = lf_parquet.collect()

#  Guardar a disco
df.write_parquet("resultado.parquet")

# en formato csv
df.write_csv("resultado.csv")

# formato IPC 
df.write_ipc("datos_locales.arrow")

**El LazyFrame no cabe en memoria RAM**: Polars procesa la consulta en streaming y escribe los datos directamente al disco sin cargar todo el dataset en memoria.

In [ ]:
#  Guardar a disco usando streaming

# (lento) Transmitir el resultado directamente a un archivo CSV
lf_parquet.sink_csv("partidos.csv")

# (rápido) formato parquet
lf_parquet.sink_parquet("partidos.parquet")

# (ultra rápido) Escribir directamente a disco en formato Apache Arrow (Feather)
lf_parquet.sink_ipc("partidos.arrow")

#### Procesamiento por Trozos (Out-of-Core / Streaming)

Lo bueno de Polars es que permite procesar archivos Parquet, CSV o IPC masivos que no caben en la RAM mediante el motor de **streaming** con un simple parámetro:

In [ ]:
# Procesa un archivo de 100 GB en una máquina con 16 GB de RAM
df_resultado = (
    pl.scan_parquet('./datos/DatasetPartidos.parquet')
    .filter(pl.col("golesVisitante") > 1)
    .group_by("temporada")
    .agg(pl.col("idPartido").len())
    .collect(engine='streaming')
)
df_resultado.head()


## Manipulación Básica con Expresiones

A diferencia de Pandas, Polars utiliza un lenguaje de expresiones declarativo: definimos las transformaciones a realizar sobre las columnas mediante `pl.col()`, y Polars las optimiza y ejecuta de forma totalmente paralela.

Ejemplos de expresiones:

- `pl.col("columna")`: referencia a una columna.
- `pl.lit(valor)`: valor literal.
- `.alias("nombre")`: nombre del resultado.
- `.cast(tipo)`: conversión de tipos.
- `.str`, `.dt` y `.list`: operaciones especializadas.

### Referencia a columnas

`pl.col()` hace referencia a una o varias columnas dentro del contexto de un DataFrame o LazyFrame.

In [ ]:
# Referenciar múltiples columnas explícitas
pl.col("EquipoLocal", "golesLocal")

# Seleccionar TODAS las columnas
pl.col("*")

# Seleccionar columnas por tipo de dato (ej. todas las flotantes)
pl.col(pl.Int64)

# Seleccionar usando expresiones regulares (regex)
pl.col("^Goles.*")

Las expresiones se ejecutan dentro de métodos que proporcionan un "contexto" específico. Así que necesitamos definir las operaciones de contexto.

### Operaciones de contexto

Hay tres contextos principales:

* Selección: `select`,  `with_columns`
* Filtrado: `filter`
* Agrupaciones y agregaciones: `group_by(..).agg()`

####  Seleccion de columnas con `select`

`select` es una transformacion que permite proyectar columnas (elegir qué columnas quedan en el resultado) y crear expresiones derivadas.

- Devuelve un nuevo objeto
- Sirve para seleccionar, reordenar y renombrar columnas con `alias`.

In [ ]:
# Ejemplo 1: seleccion simple
df_goles.select("EquipoLocal", "golesLocal").head(5)

In [ ]:
# Ejemplo 2: seleccion con expresion y alias
df_polizas.select(
    "numero_poliza",
    (pl.col("suma_asegurada") / 1000).alias("suma en miles")
).tail()

In [ ]:
# Ejemplo 3: reordenar columnas y crear derivada
df_polizas.select(
    [
        "titular",
        "provincia",
        pl.concat_str(
            [pl.col("tipo_seguro"), pl.col("estado")],
            separator="-"
        ).alias("Características")
    ]
).head(5)

In [ ]:
# Ejemplo 4: Si se trata de un lazy Frame
lf_polizas.select(
    [
        "titular",
        "provincia",
        pl.concat_str(
            [pl.col("tipo_seguro"), pl.col("estado")],
            separator="-"
        ).alias("Caracteristicas")
    ]
).show(5)    # No devuelve nada

In [ ]:
# Ejemplo 4bis: Si se trata de un lazy Frame
lf_polizas.select(
    [
        "titular",
        "provincia",
        pl.concat_str(
            [pl.col("tipo_seguro"), pl.col("estado")],
            separator="-"
        ).alias("Caracteristicas")
    ]
).head(5).collect()    # devuelve un dataframe

####  Transformación de datos con `withcolum`

Para añadir  y  modificar columnas usamos la operación `withColumn`. Mantiene todas las columnas existentes del DataFrame y añade nuevas columnas o sobrescribe las actuales.

In [ ]:
df_actualizado = df_polizas.with_columns(
    # Nueva columna calculada
    (pl.col("prima_mensual") * pl.lit(1.21)).alias("prima_final_mesual"),
    # Sobrescribir la columna nombre a mayúsculas
    pl.col("titular").str.to_uppercase(),
    # Constante
    pl.lit(2026).alias("anio_fiscal"),
    # Conversión de fecha
    pl.col("fecha_alta").str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias("fecha")    
    )

df_actualizado.select('titular', 'prima_mensual','prima_final_mesual', 'fecha_alta' , 'fecha')

#### Filtrado de filas con `filter`

Filtra el DataFrame basándose en condiciones booleanas compuestas.

**Atención a la sintaxis**: Como en Pandas, en Polars se utilizan los operadores ` &` (AND), `|` (OR) y `~` (NOT). Cada condición individual debe ir obligatoriamente entre paréntesis `()`.

In [ ]:
# Filtrado: equivalente conceptual a una máscara booleana de Pandas
df_polizas.filter(
    (pl.col("edad") >= 50) &
    (pl.col("provincia") == "Madrid")
).head()

## Casting de Tipos (cast) y Manejo de Nulos

__Conversión de tipos de datos (cast)__

In [ ]:
df_cast = df_polizas.with_columns(
    pl.col("edad").cast(pl.Float32),
    pl.col("prima_mensual").cast(pl.Int64),
)
df_cast.head(4)

__Tratamiento de valores faltantes (null)__

A diferencia de Pandas (NaN), Polars distingue claramente entre valores flotantes no válidos (NaN) y datos ausentes (null).


* `is_null()`                     
* `is_not_null()`                     
*  `fill_null(0)`                       
* `pl.coalesce(col1,col2)`             
* `drop_nulls()`                       
* `when(...).then(...).otherwise(...)` 


In [ ]:
df_nulos = pl.DataFrame(
    {"ventas": [100, None, 150, None],
     "categoria": ["A", "B", None, "A"], 
     "telefono": [123, None, 789, None],
     "movil": [None, 345, 789, 999]}
)
df_nulos

In [ ]:
df_limpio = df_nulos.with_columns(
    # Reemplazar valores nulos con un valor fijo
    pl.col("ventas").fill_null(0).alias('ventas_no_nulo'),
    # Imputación con estrategias (forward fill, backward fill, mean)
    pl.col("ventas").fill_null(strategy="forward").alias("ventas_ffill"),
    # Reemplazar nulos en texto
    pl.col("categoria").fill_null("Desconocido"),
)
df_limpio


In [ ]:
# Eliminar filas con nulos en columnas específicas
df_sin_nulos = df_nulos.drop_nulls(subset=["categoria"])
df_sin_nulos

In [ ]:
# reemplazar nulos en todas las columnas
# pl.all() equivalente a pl.col(*)
df_nulos.with_columns(
    pl.all().fill_null(0)
)

In [ ]:
# usar otra columna cuando haya nulos
df_nulos.with_columns(
    pl.coalesce(
        pl.col("telefono"),
        pl.col("movil")
    ).alias("telefono_contacto")
)

In [ ]:
# condicionales con nulos
df_nulos.with_columns(
    pl.when(pl.col("ventas").is_null())
      .then(9999)
      .otherwise(pl.col("ventas"))
      .alias("ventas_sin_nulo")
)


# Más operaciones de transformación y limpieza

Operaciones Condicionales (`when / then / otherwise`). Es el equivalente en Polars a la cláusula `np.where()` de NumPy.

In [ ]:
df_polizas.head(2)

In [ ]:

df_transformado = df_polizas.with_columns(
    pl.when(pl.col("edad") >= 40)
      .then(pl.lit("senior"))
      .otherwise(pl.lit("junior"))
      .alias("categoria")
)

df_transformado.select('edad' , 'categoria').head(10)


In [ ]:
# Renombrar y ordenar
df_transformado.rename({"provincia": "localidad"}).sort(
    "edad",
    descending=True
)



# Agrupaciones y agregaciones

`group_by` y `agg` sustituyen habitualmente a `groupby` y `agg` de Pandas. La agregación se define mediante el método `agg()`, que acepta una lista de expresiones que se evalúan en paralelo.


In [ ]:
# 1) Número de siniestros por tipo de seguro
sinestros_tipo = df_polizas.group_by('tipo_seguro').agg(
    pl.col('num_siniestros').sum().alias('total_siniestros')
)

sinestros_tipo

__Agregación Múltiple Paralela__

In [ ]:
resumen_importe_siniestros = (
    df_polizas
    .group_by("tipo_seguro")
    .agg(        
            pl.col("importe_siniestros").sum().alias("sum"),
            pl.col("importe_siniestros").mean().alias("mean"),
            pl.col("importe_siniestros").max().alias("max"),
    )
)

resumen_importe_siniestros.sort("max", descending=True)


__Funciones ventana (Window Functions) con `.over()`__

Permiten realizar cálculos agrupados o acumulados **sin reducir el número de filas** del DataFrame original (equivalente  `transform()` en Pandas, pero de ejecución mucho más rápida).


In [ ]:
# usar transventanas form para comparar cada póliza
# con la media de su tipo de seguro
polizas = df_polizas.with_columns(
    pl.col("prima_mensual")
      .mean()
      .over("tipo_seguro")
      .alias("prima_media_tipo")
)
polizas.sort("tipo_seguro", descending=True)



In [ ]:
df_con_metricas = df_polizas.with_columns(
    # Calcular importe prima mensual por tipo se seguro
    pl.col("prima_mensual").sum().over("tipo_seguro").alias("total_tipo_seguro"),

    # Asignar un número de fila secuencial dentro de cada tipo de 
    # seguro ordenado por prima_mensual
    pl.col("prima_mensual").rank(descending=True).over("tipo_seguro").alias("ranking")
    
    ).sort('tipo_seguro', 'ranking', descending = False)

df_con_metricas.select('tipo_seguro', "total_tipo_seguro",'prima_mensual' ,'ranking')

__pivot() (Resumen tabular / Tablas dinámicas)__

Transforma valores únicos de una columna en múltiples columnas independientes.

Se usa la sintaxis:

```python
   # pivot_table
   # Si hay valores duplicados es necesario agregar con `aggfun`
   df.pivot_table(on = ...,  # valores que se convierten en columnas
                  index=..., # Columna que actuará como índice de filas
                  values=..., # Columna a agregar  
                  aggregate_function=... # Función de agregación
                  )
```

In [ ]:
# 1) Prima por número de póliza y estado (muestra la información de otra manera. )
# El número de póliza y estod de la póliza no pueden repetrise en el dataset.
tabla_pivot = df_polizas.pivot(
    index='numero_poliza',
    on='estado',
    values='prima_mensual'
)
tabla_pivot.head(10)

In [ ]:
# 2) Número de pólizas con titular y estado de la póliza
tabla_pivot = polizas.pivot(
    index='titular',
    on='estado',
    values='prima_mensual', 
    aggregate_function = 'len'   # count
)

tabla_pivot.head(10)

In [ ]:
# 4) Prima media de las polizas por provincia y tipo de seguro
df_polizas.pivot(
    index='provincia',
    on='tipo_seguro',
    values='prima_mensual',
    aggregate_function='mean'
)


# Joins y concatenación


## Joins

Los **joins** nos permiten combinar dos DataFrames basándonos en una o más columnas comunes. Hay varios tipos:

Tipos habituales de join:

- `inner`: únicamente claves presentes en ambos DataFrames.
- `left`: conserva todas las filas del DataFrame izquierdo.
- `right`: conserva todas las filas del DataFrame derecho.
- `full`: conserva todas las claves.
- `semi`: filas del izquierdo que tienen correspondencia.
- `anti`: filas del izquierdo sin correspondencia.

In [ ]:
# Ejemplos
empleados = pl.DataFrame({
    "empleado_id": [1, 2, 3, 4],
    "nombre": ["Ana", "Luis", "Marta", "Pablo"],
    "departamento_id": [10, 20, 10, 30],
})

departamentos = pl.DataFrame({
    "departamento_id": [10, 20, 30],
    "departamento": ["IT", "Ventas", "Finanzas"],
    "director_id": [3, 4, 3]
})


In [ ]:

empleados.join(
    departamentos,
    on="departamento_id",
    how="left"
)


In [ ]:
empleados.join(
    departamentos,
    left_on="empleado_id",
    right_on="director_id",
    how="left"
)

## Concatenación Vertical y Horizontal con concat()

Polars permite unir múltiples DataFrames a lo largo de un eje usando `pl.concat()`

__Concatenación Vertical (Añadir Filas)__

* ` how="vertical"`: Exige que las columnas y tipos sean idénticos.
* `how="diagonal"`: Alinea por nombre de columna e inserta null en las columnas no compartidas.

In [ ]:

df_enero = pl.DataFrame({
    "mes": ["enero", "enero"],
    "ventas": [100, 150],
})

df_febrero = pl.DataFrame({
    "mes": ["febrero", "febrero"],
    "ventas": [120, 180],
    "costes": [14, 52]
})

pl.concat([df_enero, df_febrero], how="diagonal")


__Concatenación Horizontal (Añadir Columnas)__

Une DataFrames posicionando sus columnas lado a lado basándose en la posición física de las filas (sin realizar búsqueda por claves).

In [ ]:
df_base = pl.DataFrame({"id": [1, 2, 3]})
df_metricas = pl.DataFrame({"score": [88, 92, 79], "rango": ["B", "A", "C"]})

pl.concat([df_base, df_metricas], how="horizontal_extend")

## Referencias

* Documentación [https://docs.pola.rs/user-guide/getting-started/+(https://docs.pola.rs/user-guide/getting-started/)]

* Python API reference [https://docs.pola.rs/api/python/stable/reference/index.html](https://docs.pola.rs/api/python/stable/reference/index.html)